In [ ]:
# PARAMETERS — edit this cell only.
from __future__ import annotations

import json
import os
import re
import sys
from pathlib import Path


def locate_bundle() -> Path:
    starts = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for candidate in starts:
        if (candidate / "data" / "example_tiles.csv").is_file():
            return candidate
        nested = candidate / "tasks" / "260731_2_mllm_notebook_bundle"
        if (nested / "data" / "example_tiles.csv").is_file():
            return nested
    return Path.cwd().resolve()


BUNDLE_DIR = locate_bundle()
PROJECT_DIR = next(
    (candidate for candidate in [BUNDLE_DIR, *BUNDLE_DIR.parents]
     if (candidate / "experiments").is_dir() and (candidate / "scripts").is_dir()),
    BUNDLE_DIR,
)
sys.path.append(str(PROJECT_DIR))

# Required for live tile loading. Leave empty for the included six-tile examples.
COUNTING_DATA_ROOT = os.environ.get("COUNTING_DATA_ROOT", "")
TASKS = [
    "malaria/trophozoite", "malaria/difficult", "malaria/red_blood_cell",
    "birds/bird", "dota/tennis_court", "dota/harbor", "dota/ship",
    "dota/plane", "dota/small_vehicle", "kenyan_wildlife/giraffe",
    "dota/helicopter", "dota/storage_tank",
]
MODEL_LABEL = "Qwen3-VL-8B-Instruct"
BATCH_SIZE = 8
MAX_CHOICE = 300
GPU_MEMORY_UTILIZATION = 0.85
LABEL_ROOT = PROJECT_DIR / "outputs" / "mllm_all_tiles_all_models"
PROPRIETARY_ROOT = PROJECT_DIR / "outputs" / "proprietary_all_tiles_260710"
RUN_ROOT = PROJECT_DIR / "outputs" / "runs"
EXAMPLE_CSV = BUNDLE_DIR / "data" / "example_tiles.csv"
OPEN_SOURCE_CSV = PROJECT_DIR / "outputs" / "runs" / "open_source_all_tiles_current.csv" if (PROJECT_DIR / "outputs" / "runs" / "open_source_all_tiles_current.csv").is_file() else BUNDLE_DIR / "data" / "open_source_reference_sample.csv"
PROPRIETARY_CSV = PROJECT_DIR / "outputs" / "runs" / "proprietary_all_tiles_current.csv" if (PROJECT_DIR / "outputs" / "runs" / "proprietary_all_tiles_current.csv").is_file() else BUNDLE_DIR / "data" / "proprietary_reference_sample.csv"
OPENROUTER_KEY_SELECTOR = "OPENROUTER_API_KEY"

try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except NameError:
    pass


# 00 — Setup and configuration

This short notebook is the map for the bundle. The next notebooks are independent: each repeats a single parameter cell so it can be opened and run on its own. Set `COUNTING_DATA_ROOT` to the directory containing the counting dataset when you want live inference. The included CSVs and six PNG tiles make inspection and analysis useful even without the raw dataset.


In [ ]:
print({
    "bundle_dir": str(BUNDLE_DIR), "project_dir": str(PROJECT_DIR),
    "tasks": len(TASKS), "model": MODEL_LABEL,
    "counting_data_root_set": bool(COUNTING_DATA_ROOT),
    "open_source_csv": OPEN_SOURCE_CSV.is_file(), "proprietary_csv": PROPRIETARY_CSV.is_file(),
})


## What runs where

The open-source notebook assumes one visible GPU and calls vLLM or Transformers directly. The proprietary notebook calls OpenRouter over HTTPS and uses a bounded thread pool; it does not need a GPU for API calls. The remaining notebooks are analysis/export stages and can run without loading a model. Live paid requests are opt-in through a parameter at the top of that notebook.


In [ ]:
from mu.visv2.figure import Figure

stage_names = ["setup", "open-source\ninference", "proprietary\nAPI", "extract +\nverify", "g_fixed\nstores", "timing +\ncost", "exports"]
fig = Figure(figsize=(11, 2.8), fontfamily="sans-serif", fontsize=11)
panel = fig.panels[0]
for index, stage in enumerate(stage_names):
    panel.p.scatter(index, 0, s=850, color="#2563eb", edgecolor="white", linewidth=2, zorder=3)
    panel.p.text(index, 0, str(index), color="white", ha="center", va="center", weight="bold")
    panel.p.text(index, -0.18, stage, ha="center", va="top", fontsize=10)
    if index < len(stage_names) - 1:
        panel.p.annotate("", xy=(index + 0.78, 0), xytext=(index + 0.22, 0), arrowprops={"arrowstyle": "->", "color": "#94a3b8", "lw": 2})
panel.p.set_xlim(-0.6, len(stage_names) - 0.4)
panel.p.set_ylim(-0.5, 0.45)
panel.p.axis("off")
fig.add_suptitle("Notebook path: frozen tile → count → analysis", size=15)
fig.show()


The figure is deliberately small: it is a quick orientation check, not a replacement for the stage notebooks. No cache, model, API key, or external execution state is written by this setup notebook.
